# 02 - Preprocesamiento
Deducciones de la etapa 1 - EDA

En el notebook `01_eda.ipynb` vimos que el dataset no tiene nulos ni duplicados, que las variables ya vienen entre -1 y 1 y que hay muchas variables correlacionadas.

In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import skew

from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import KFold, GroupKFold, cross_val_score

SEED = 42
pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 250)

## 1. Carga de datos
Además de `X` y `y`, cargamos los archivos `subject_train.txt` y `subject_test.txt`, que indican a qué persona pertenece cada ventana.

In [4]:
base = "../data"

def leer(archivo):
    return pd.read_csv(f"{base}/{archivo}", sep=r"\s+", header=None)

X_train = leer("train/X_train.txt")
X_test = leer("test/X_test.txt")
y_train = leer("train/y_train.txt")[0]
y_test = leer("test/y_test.txt")[0]
s_train = leer("train/subject_train.txt")[0]
s_test = leer("test/subject_test.txt")[0]
labels = leer("activity_labels.txt").set_index(0)[1]
nombres = leer("features.txt")[1]

print("X_train:", X_train.shape, " X_test:", X_test.shape)
print("Nombres de variables repetidos:", nombres.duplicated().sum())
print(nombres[nombres.duplicated(keep=False)].head(6).tolist())

X_train: (7352, 561)  X_test: (2947, 561)
Nombres de variables repetidos: 84
['fBodyAcc-bandsEnergy()-1,8', 'fBodyAcc-bandsEnergy()-9,16', 'fBodyAcc-bandsEnergy()-17,24', 'fBodyAcc-bandsEnergy()-25,32', 'fBodyAcc-bandsEnergy()-33,40', 'fBodyAcc-bandsEnergy()-41,48']


Encontramos que `features.txt` tiene 84 nombres repetidos. Son las variables `bandsEnergy`, que se calculan para los ejes X, Y y Z pero quedaron con el mismo nombre. Como pandas necesita nombres de columna únicos, les agregamos un sufijo `_1`, `_2` a las repetidas.

In [5]:
repeticion = nombres.groupby(nombres).cumcount()
nombres_unicos = np.where(repeticion > 0, nombres + "_" + repeticion.astype(str), nombres)
print("Nombres únicos ahora:", len(set(nombres_unicos)), "de", len(nombres_unicos))

X_train.columns = nombres_unicos
X_test.columns = nombres_unicos

Nombres únicos ahora: 561 de 561


## 2. Nulos y duplicados

In [7]:
# No es necesario realizar alguna imputación de valores nulos, ya que no hay nulos en el dataset.
print("Nulos en train:", X_train.isna().sum().sum(), "| Nulos en test:", X_test.isna().sum().sum())
print("Filas duplicadas en train:", X_train.duplicated().sum(), "| en test:", X_test.duplicated().sum())
print("Columnas constantes:", (X_train.std() == 0).sum())

Nulos en train: 0 | Nulos en test: 0
Filas duplicadas en train: 0 | en test: 0
Columnas constantes: 0


## 3. Codificación de la etiqueta